# NexTwin AI — Industrial Digital Twin Platform
## Notebook 03: Industrial Feature Engineering & Health Scores

### Objectives
1. **Load Cleaned Datasets**: Load processed datasets from `datasets/processed/`.
2. **Calculate Core Industrial Features**:
   - **Machine Health Score**: Normalized composite health index (0 to 100).
   - **Failure Risk Index**: Logistic stress-based failure probability (0 to 1).
   - **Energy Efficiency Score**: Ratio of workload throughput to energy drawn.
   - **Operational Efficiency Score**: Combination of availability, performance, and quality.
   - **Downtime Risk**: Predictive risk indicator of sudden maintenance shutdown.
   - **Remaining Useful Life (RUL) Indicators**: Engine degradation tracking for NASA Turbofan.
   - **Bottleneck Severity Index**: Detect production line congestion points.
   - **Throughput Efficiency**: Work order compliance metric.
   - **Factory Health Score**: Unified enterprise-level dashboard health index.
3. **Export Engineered Datasets**: Save data to `datasets/processed/` for model training.

In [1]:
import os
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Processed Datasets
We load our cleaned files from `datasets/processed/`.

In [2]:
PROCESSED_DIR = os.path.join("..", "datasets", "processed")

df_pm = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_ai4i_predictive_maintenance.csv"))
df_energy = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_energy_consumption.csv"))
df_util = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_machine_utilization.csv"))
df_prod = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_production_metrics.csv"))
df_factory = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_synthetic_factory_data.csv"))
df_nasa_train = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_nasa_train_fd001.csv"))

print("Processed datasets loaded.")

Processed datasets loaded.


## 2. Machine Health & Failure Risk Features (AI4I)
We calculate the **Machine Health Score** and **Failure Risk Index** for the AI4I predictive maintenance dataset.

- **Machine Health Score (MHS)**: Normal operating values are mapped between 0 and 100. Lower score means higher wear or thermal/torque stress.
  $$MHS = 100 - \left(0.3 \times \frac{\text{tool\_wear}}{240} + 0.4 \times \frac{|\text{torque} - 40|}{40} + 0.3 \times \frac{\text{process\_temperature} - \text{air\_temperature}}{15}\right) \times 100$$
  Values are clipped to range $[0, 100]$.

- **Failure Risk Index (FRI)**: Sigmoid function mapping cumulative stress:
  $$\text{Stress} = 2.0 \times \text{wear\_ratio} + 1.5 \times \text{torque\_dev} + 3.0 \times \text{temp\_diff}$$
  $$FRI = \frac{1}{1 + e^{-(\text{Stress} - 2.5)}}$$

In [3]:
# 1. Tool wear ratio
tool_wear_ratio = np.clip(df_pm['tool_wear'] / 240.0, 0, 1)
# 2. Torque deviation from ideal mean of 40
torque_dev = np.clip(np.abs(df_pm['torque'] - 40.0) / 40.0, 0, 1)
# 3. Process vs Air Temperature difference deviation (target differential is 10K)
temp_diff = df_pm['process_temperature'] - df_pm['air_temperature']
temp_diff_dev = np.clip(np.abs(temp_diff - 10.0) / 10.0, 0, 1)

# Composite Health Score
df_pm['machine_health_score'] = 100.0 - (0.3 * tool_wear_ratio + 0.4 * torque_dev + 0.3 * temp_diff_dev) * 100.0
df_pm['machine_health_score'] = np.clip(df_pm['machine_health_score'], 0.0, 100.0)

# Failure Risk Index
stress = 2.0 * tool_wear_ratio + 1.5 * torque_dev + 3.0 * temp_diff_dev
df_pm['failure_risk_index'] = 1.0 / (1.0 + np.exp(-stress + 2.5))

print("AI4I Health Features calculated. Samples:")
display(df_pm[['machine_health_score', 'failure_risk_index', 'machine_failure']].head(5))

AI4I Health Features calculated. Samples:

,machine_health_score,failure_risk_index,machine_failure
0,95.700,0.095782,0
1,91.825,0.110195,0
2,88.775,0.120698,0
3,97.425,0.090882,0
4,97.375,0.093215,0


## 3. Energy Efficiency and Waste Features
We construct metrics for the Energy Consumption dataset:
- **Energy Efficiency Score**: Ratio of inverse thermal load, scaled 0 to 100.
- **Energy Waste %**: Relative heat/cooling load deviation above the baseline efficient envelope (25th percentile of load).

In [4]:
# Combined Thermal Load
total_load = df_energy['heating_load'] + df_energy['cooling_load']
df_energy['total_load'] = total_load

# Energy Efficiency Score
df_energy['energy_efficiency_score'] = 100.0 * (total_load.min() / total_load)

# Energy Waste %
baseline_load = total_load.quantile(0.25)
df_energy['energy_waste_pct'] = np.clip(((total_load - baseline_load) / baseline_load) * 100.0, 0.0, 500.0)

# Energy Optimization Score (inverse of waste)
df_energy['energy_optimization_score'] = 100.0 - df_energy['energy_waste_pct']
df_energy['energy_optimization_score'] = np.clip(df_energy['energy_optimization_score'], 0.0, 100.0)

print("Energy metrics generated. Samples:")
display(df_energy[['total_load', 'energy_efficiency_score', 'energy_waste_pct', 'energy_optimization_score']].head(5))

Energy metrics generated. Samples:


,total_load,energy_efficiency_score,energy_waste_pct,energy_optimization_score
0,36.88,49.822668,28.278261,71.721739
1,36.88,49.822668,28.278261,71.721739
2,36.88,49.822668,28.278261,71.721739
3,36.88,49.822668,28.278261,71.721739
4,49.12,37.407573,70.852174,29.147826


## 4. Remaining Useful Life (RUL) Indicators (NASA Turbofan)
For time-to-failure prediction on the NASA C-MAPSS dataset, we calculate the remaining useful life (in cycles) for each engine at each time index:
$$RUL_{t, u} = MaxCycles_u - t$$
We also build rolling sensor statistics (rolling mean and rolling standard deviation) to capture deterioration trends.

In [5]:
# Calculate max cycles for each unit
max_cycles = df_nasa_train.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycles.columns = ['unit_number', 'max_cycles']

# Merge back and calculate RUL
df_nasa_train = df_nasa_train.merge(max_cycles, on='unit_number', how='left')
df_nasa_train['rul'] = df_nasa_train['max_cycles'] - df_nasa_train['time_in_cycles']
df_nasa_train = df_nasa_train.drop(columns=['max_cycles'])

# Add rolling metrics for trending sensors (sensor_2, sensor_7, sensor_11)
rolling_cols = ['sensor_2', 'sensor_7', 'sensor_11']
for col in rolling_cols:
    # Rolling mean (window of 5 cycles)
    df_nasa_train[f'{col}_roll_mean_5'] = df_nasa_train.groupby('unit_number')[col].transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    # Rolling standard deviation
    df_nasa_train[f'{col}_roll_std_5'] = df_nasa_train.groupby('unit_number')[col].transform(lambda x: x.rolling(window=5, min_periods=1).std().fillna(0))

print("NASA Turbofan RUL features engineered. Samples:")
display(df_nasa_train[['unit_number', 'time_in_cycles', 'rul', 'sensor_2_roll_mean_5', 'sensor_2_roll_std_5']].head(5))

NASA Turbofan RUL features engineered. Samples:


,unit_number,time_in_cycles,rul,sensor_2_roll_mean_5,sensor_2_roll_std_5
0,1,1,191,641.820000,0.000000
1,1,2,190,641.985000,0.233345
2,1,3,189,642.106667,0.267644
3,1,4,188,642.167500,0.250117
4,1,5,187,642.208000,0.234776


## 5. Manufacturing Operations, Bottlenecks, & Factory Health Scores
Here we combine the utilization and production logs, map machines to production lines, and calculate industrial operational efficiency, bottlenecks, and downtime risks:
- **Machine to Line Mapping**:
  - Machine `M_001` and `M_002` map to `Line_A`.
  - Machine `M_003` maps to `Line_B`.
- **Throughput Efficiency**: $\text{Actual Quantity} / \text{Target Quantity}$.
- **Operational Efficiency Score**: $\text{OEE} \times \text{Throughput Efficiency} \times \text{Utilization}$.
- **Bottleneck Severity Index**: Scale of 0-100 indicating congestion based on high utilization, low operational efficiency, and high defect rate.
- **Downtime Risk**: Risk of unexpected machine failure due to stress.
- **Factory Health Score**: Combined corporate score aggregating machine health and operational efficiency.

In [6]:
# Create machine to line mapping
m_to_l = {"M_001": "Line_A", "M_002": "Line_A", "M_003": "Line_B"}
df_util['line_id'] = df_util['machine_id'].map(m_to_l)

# Merge utilization and production metrics on timestamp and line_id
df_mfg = pd.merge(df_util, df_prod, on=['timestamp', 'line_id'], how='inner')

# Merge factory acoustic/physical sensor logs as well
df_mfg = pd.merge(df_mfg, df_factory, on=['timestamp', 'machine_id'], how='inner')

# Calculate Throughput Efficiency
df_mfg['throughput_efficiency'] = df_mfg['actual_quantity'] / df_mfg['target_quantity']
df_mfg['throughput_efficiency'] = np.clip(df_mfg['throughput_efficiency'], 0.0, 2.0)

# Calculate Operational Efficiency Score (OES)
df_mfg['operational_efficiency_score'] = df_mfg['oee'] * df_mfg['throughput_efficiency'] * (df_mfg['utilization_rate'] / 100.0) * 100.0
df_mfg['operational_efficiency_score'] = np.clip(df_mfg['operational_efficiency_score'], 0.0, 100.0)

# Calculate Downtime Risk (probability 0 to 1 based on stress indicators)
# Stress increases with high temperature, vibration, and low cycle time
mfg_stress = (df_mfg['vibration_mm_s'] / 4.0) + (df_mfg['temperature_c'] - 60.0) / 15.0 + (df_mfg['defect_count'] / 5.0)
df_mfg['downtime_risk'] = 1.0 / (1.0 + np.exp(-mfg_stress + 2.0))

# Calculate Bottleneck Severity Index (0 to 100)
# Higher when utilization is high but OEE is low and defects are high
df_mfg['bottleneck_severity_index'] = (df_mfg['utilization_rate'] / 100.0) * (1.0 - df_mfg['oee']) * (1.0 + df_mfg['defect_count'] / (df_mfg['actual_quantity'] + 1)) * 100.0
df_mfg['bottleneck_severity_index'] = np.clip(df_mfg['bottleneck_severity_index'], 0.0, 100.0)

# Calculate Machine-level Health Score (simulated dynamically using sensor readings)
sensor_health_ratio = 1.0 - (0.4 * (df_mfg['vibration_mm_s'] / 5.0) + 0.3 * (df_mfg['temperature_c'] / 90.0) + 0.3 * (df_mfg['noise_level_db'] / 95.0))
df_mfg['machine_health_score'] = np.clip(sensor_health_ratio * 100.0, 0.0, 100.0)

# Calculate Factory Health Score (FHS) - aggregated over time
# FHS = mean(Machine Health) * mean(Throughput Efficiency) * (1 - mean(Downtime Risk))
# For timeseries representation, we make a rolling factory-wide health score
df_mfg['factory_health_score'] = df_mfg['machine_health_score'] * df_mfg['throughput_efficiency'] * (1.0 - df_mfg['downtime_risk'])
df_mfg['factory_health_score'] = np.clip(df_mfg['factory_health_score'], 0.0, 100.0)

print("Manufacturing integrated features engineered. Shape:", df_mfg.shape)
display(df_mfg[['machine_id', 'throughput_efficiency', 'operational_efficiency_score', 'downtime_risk', 'bottleneck_severity_index', 'factory_health_score']].head(5))

Manufacturing integrated features engineered. Shape: (6483, 25)


,machine_id,throughput_efficiency,operational_efficiency_score,downtime_risk,bottleneck_severity_index,factory_health_score
0,M_001,0.905917,52.432246,0.601767,21.031840,16.091625
1,M_001,1.018667,70.337416,0.555438,8.361162,19.461537
2,M_001,0.892333,59.808506,0.407454,16.965298,23.866202
3,M_001,0.804750,52.466208,0.331923,24.996427,20.700132
4,M_001,0.742750,43.372642,0.319842,24.630008,21.366408


## 6. Save Engineered Datasets
We export the final datasets with all engineered industrial features.

In [7]:
df_pm.to_csv(os.path.join(PROCESSED_DIR, "engineered_machine_health.csv"), index=False)
df_energy.to_csv(os.path.join(PROCESSED_DIR, "engineered_energy.csv"), index=False)
df_mfg.to_csv(os.path.join(PROCESSED_DIR, "engineered_mfg_bottleneck.csv"), index=False)
df_nasa_train.to_csv(os.path.join(PROCESSED_DIR, "engineered_nasa_train_fd001.csv"), index=False)

print("Engineered datasets saved successfully. Ready for machine learning modeling.")

Engineered datasets saved successfully. Ready for machine learning modeling.
